In [1]:
!pip install nltk

In [2]:
import nltk
from nltk import CFG, ChartParser

# 1. Load the CFG derived from the dataset
grammar = CFG.fromstring("""
    S -> Greeting Query | Query
    Greeting -> Salutation Title Punc | TimeGreeting Title Punc
    Query -> MarksQuery | RegQuery | WaiverQuery | FinanceQuery | GradQuery

    Salutation -> 'Hello' | 'Dear' | 'Hi'
    TimeGreeting -> 'Good_morning'
    Title -> 'COD' | 'Dean'
    Punc -> ','

    MarksQuery -> Pronoun Action_Marks Pronoun Noun_Marks Prep Target_Marks
    Pronoun -> 'I' | 'my'
    Action_Marks -> 'am_inquiring_about' | 'am_reporting'
    Noun_Marks -> 'missing_marks' | 'a_missing_mark'
    Prep -> 'for'
    Target_Marks -> 'unit_CCS_3104' | 'unit_SMA_2101'

    RegQuery -> Question_Word Det Noun_Reg Prep Target_Reg TimeFrame
    Question_Word -> 'what_is'
    Det -> 'the' | 'a'
    Noun_Reg -> 'deadline'
    Target_Reg -> 'unit_registration' | 'special_exam'
    TimeFrame -> 'this_semester'

    WaiverQuery -> Pronoun Action_Waiver Det Noun_Waiver Prep Det Target_Waiver
    Action_Waiver -> 'would_like_to_request'
    Noun_Waiver -> 'prerequisite_waiver'
    Target_Waiver -> 'Theory_of_Computation_unit'

    FinanceQuery -> Question_How Action_Apply Det Noun_Finance Prep Det Time_Year
    Question_How -> 'How_do_I'
    Action_Apply -> 'apply_for'
    Noun_Finance -> 'University_Bursary'
    Time_Year -> 'current_academic_year'

    GradQuery -> Pronoun Action_Request Det Noun_Grad Target_Purpose
    Action_Request -> 'am_requesting'
    Noun_Grad -> 'letter_of_completion'
    Target_Purpose -> 'to_facilitate_my_job_application'
""")


In [3]:
parser = ChartParser(grammar)

# 2. Define a batch of 10 test queries (9 valid, 1 invalid to test fallback)
test_queries = [
    "Hello COD , I am_inquiring_about my missing_marks for unit_CCS_3104",
    "Good_morning Dean , what_is the deadline for unit_registration this_semester",
    "Dear COD , I would_like_to_request a prerequisite_waiver for the Theory_of_Computation_unit",
    "How_do_I apply_for the University_Bursary for the current_academic_year",
    "I am_requesting a letter_of_completion to_facilitate_my_job_application",
    "Hi Dean , I am_reporting my a_missing_mark for unit_SMA_2101",
    "what_is a deadline for special_exam this_semester",
    "Hello Dean , I would_like_to_request a prerequisite_waiver for the Theory_of_Computation_unit",
    "Dear COD , I am_inquiring_about my missing_marks for unit_SMA_2101",
    "Hello COD , where is the nearest cafeteria on campus" # Out of grammar query
]

In [4]:
# 3. Iterate through the batch and evaluate each query
print(f"{'QUERY STATUS':<15} | {'DETECTED INTENT':<35} | {'ORIGINAL INPUT'}")
print("-" * 100)

for query in test_queries:
    tokens = query.split()

    try:
        parse_trees = list(parser.parse(tokens))

        if parse_trees:
            tree = parse_trees[0]
            intent_found = False

            # Traverse the tree to extract the specific intent
            for subtree in tree.subtrees():
                label = subtree.label()
                if label in ['MarksQuery', 'RegQuery', 'WaiverQuery', 'FinanceQuery', 'GradQuery']:
                    print(f"{'[ SUCCESS ]':<15} | {label:<35} | {query}")
                    intent_found = True
                    break

            # Failsafe if parsed but no tracked intent matched
            if not intent_found:
                print(f"{'[ PARSED ]':<15} | {'Unknown Intent (Check Logic)':<35} | {query}")

        else:
            print(f"{'[ REJECTED ]':<15} | {'Routing to AI Fallback...':<35} | {query}")

    except ValueError:
        # This catches words that do not exist in the defined grammar's lexicon
        print(f"{'[ LEX ERROR ]':<15} | {'Routing to AI Fallback...':<35} | {query}")

print("-" * 100)
print("Batch testing complete.")

QUERY STATUS    | DETECTED INTENT                     | ORIGINAL INPUT
----------------------------------------------------------------------------------------------------
[ SUCCESS ]     | MarksQuery                          | Hello COD , I am_inquiring_about my missing_marks for unit_CCS_3104
[ SUCCESS ]     | RegQuery                            | Good_morning Dean , what_is the deadline for unit_registration this_semester
[ SUCCESS ]     | WaiverQuery                         | Dear COD , I would_like_to_request a prerequisite_waiver for the Theory_of_Computation_unit
[ SUCCESS ]     | FinanceQuery                        | How_do_I apply_for the University_Bursary for the current_academic_year
[ SUCCESS ]     | GradQuery                           | I am_requesting a letter_of_completion to_facilitate_my_job_application
[ SUCCESS ]     | MarksQuery                          | Hi Dean , I am_reporting my a_missing_mark for unit_SMA_2101
[ SUCCESS ]     | RegQuery                        